# Лабораторная работа: Cython — практические задания

**Цель:** научиться применять Cython для ускорения Python-кода.
Каждое задание содержит:
- Описание задачи
- Шаблон кода с `# TODO`
- Тест для проверки корректности
- Бенчмарк для измерения ускорения

---
## Задания
| # | Тема | Сложность |
|---|------|-----------|
| 1 | `cdef` переменные — ускорение простого цикла | Начальный |
| 2 | `cpdef` функция — факториал без PyObject | Начальный |
| 3 | Typed memoryview — скалярное произведение | Средний |
| 4 | 2D memoryview + `nogil` — матричное умножение | Средний |
| 5 | Полная оптимизация — свёртка сигнала | Продвинутый |

---
**Требования:** Cython + C-компилятор (MSVC на Windows, gcc на Linux/macOS)

```bash
pip install cython numpy matplotlib
```

---
## Подготовка

In [ ]:
%load_ext Cython
import numpy as np
import time

def bench(fn, *args, repeats=7):
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best * 1000  # ms

print('Готово. Можно приступать к заданиям.')

---
# Задание 1 — `cdef` переменные
**Сложность:** Начальный | **Тема:** статические типы

## Задача
Реализуйте функцию `sum_of_squares(n)`, которая вычисляет
сумму квадратов чисел от 0 до n-1:

```
sum_of_squares(5) = 0^2 + 1^2 + 2^2 + 3^2 + 4^2 = 30
```

## Что нужно сделать
1. Заполните `# TODO` в ячейке ниже
2. Объявите переменные `i` и `s` через `cdef` с правильными типами
3. Запустите тест и бенчмарк

## Подсказка
```cython
cdef int i          # C int на стеке
cdef long long s    # C long long (для больших чисел)
```

In [ ]:
# Эталонная реализация на чистом Python
def py_sum_of_squares(n: int) -> int:
    s = 0
    for i in range(n):
        s += i * i
    return s

print('Python:', py_sum_of_squares(5))  # 30

In [ ]:
%%cython --quiet
# ЗАДАНИЕ 1 — заполните TODO

def cy_sum_of_squares(int n):
    """
    Вычислите сумму квадратов 0^2 + 1^2 + ... + (n-1)^2
    Используйте cdef для переменных i и s
    """
    # TODO: объявите переменные через cdef
    # cdef ... i
    # cdef ... s = 0

    for i in range(n):
        s += i * i
    return s

In [ ]:
# ТЕСТ — должен пройти без ошибок
assert cy_sum_of_squares(0)  == 0,   'Ошибка: n=0'
assert cy_sum_of_squares(1)  == 0,   'Ошибка: n=1'
assert cy_sum_of_squares(5)  == 30,  'Ошибка: n=5'
assert cy_sum_of_squares(10) == 285, 'Ошибка: n=10'
print('Все тесты прошли!')

In [ ]:
# БЕНЧМАРК
N = 1_000_000
t_py = bench(py_sum_of_squares, N)
t_cy = bench(cy_sum_of_squares, N)
print(f'sum_of_squares(N={N:,})')
print(f'  Python : {t_py:.2f} ms')
print(f'  Cython : {t_cy:.2f} ms')
print(f'  Speedup: {t_py/t_cy:.1f}x')
print()
if t_py / t_cy < 10:
    print('Подсказка: убедитесь что объявили cdef для i и s')
else:
    print('Отлично! Вы получили значительное ускорение.')

---
# Задание 2 — `cpdef` функция
**Сложность:** Начальный | **Тема:** def / cdef / cpdef

## Задача
Реализуйте функцию `factorial(n)` через `cpdef`.
Функция должна быть доступна из Python, но вызываться как C-функция из Cython.

```
factorial(0) = 1
factorial(5) = 120
factorial(10) = 3628800
```

## Что нужно сделать
1. Используйте `cpdef` вместо `def`
2. Укажите тип возвращаемого значения и аргумента
3. Объявите локальную переменную через `cdef`

## Подсказка
```cython
cpdef long long factorial(int n):  # cpdef = C + Python
    cdef long long result = 1
    cdef int i
    ...
```

In [ ]:
# Эталон на Python
def py_factorial(n: int) -> int:
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print('Python factorial(10):', py_factorial(10))  # 3628800

In [ ]:
%%cython --quiet
# ЗАДАНИЕ 2 — заполните TODO

# TODO: замените def на cpdef с правильными типами
def cy_factorial(n):
    """
    Вычислите n! = 1 * 2 * 3 * ... * n
    Используйте cpdef и cdef для переменных
    """
    # TODO: объявите result и i через cdef
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

In [ ]:
# ТЕСТ
assert cy_factorial(0)  == 1,       'Ошибка: 0!'
assert cy_factorial(1)  == 1,       'Ошибка: 1!'
assert cy_factorial(5)  == 120,     'Ошибка: 5!'
assert cy_factorial(10) == 3628800, 'Ошибка: 10!'
print('Все тесты прошли!')
print('Тип функции:', type(cy_factorial))

In [ ]:
# БЕНЧМАРК
N = 20
t_py = bench(py_factorial, N)
t_cy = bench(cy_factorial, N)
print(f'factorial({N})')
print(f'  Python : {t_py:.4f} ms')
print(f'  Cython : {t_cy:.4f} ms')
print(f'  Speedup: {t_py/t_cy:.1f}x')

---
# Задание 3 — Typed Memoryview
**Сложность:** Средний | **Тема:** typed memoryviews

## Задача
Реализуйте скалярное произведение двух векторов через typed memoryview.

```
dot([1, 2, 3], [4, 5, 6]) = 1*4 + 2*5 + 3*6 = 32
```

## Что нужно сделать
1. Используйте `double[::1]` как тип аргументов (C-contiguous 1D)
2. Объявите `i` и `s` через `cdef`
3. Добавьте директивы `boundscheck=False` и `wraparound=False`

## Подсказка
```cython
# cython: boundscheck=False
# cython: wraparound=False

def dot(double[::1] a, double[::1] b):
    cdef int n = a.shape[0]
    cdef int i
    cdef double s = 0.0
    for i in range(n):
        s += a[i] * b[i]  # -> *(ptr + i*stride)
    return s
```

**Важно:** передавать нужно `np.ascontiguousarray(arr, dtype=np.float64)`

In [ ]:
import numpy as np

# Эталон на Python
def py_dot(a, b):
    s = 0.0
    for i in range(len(a)):
        s += a[i] * b[i]
    return s

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print('Python dot:', py_dot(a, b))  # 32.0
print('NumPy dot: ', np.dot(a, b))  # 32.0

In [ ]:
%%cython --quiet
# cython: boundscheck=False
# cython: wraparound=False
# ЗАДАНИЕ 3 — заполните TODO

def cy_dot(a, b):  # TODO: добавьте типы аргументов double[::1]
    """
    Скалярное произведение двух векторов.
    Используйте typed memoryview double[::1]
    """
    # TODO: объявите n, i, s через cdef
    n = len(a)
    s = 0.0
    for i in range(n):
        s += a[i] * b[i]
    return s

In [ ]:
# ТЕСТ
import numpy as np
a = np.ascontiguousarray([1.0, 2.0, 3.0], dtype=np.float64)
b = np.ascontiguousarray([4.0, 5.0, 6.0], dtype=np.float64)

result = cy_dot(a, b)
assert abs(result - 32.0) < 1e-10, f'Ошибка: {result} != 32.0'

# Тест на большом векторе
rng = np.random.default_rng(42)
a_big = np.ascontiguousarray(rng.random(10000), dtype=np.float64)
b_big = np.ascontiguousarray(rng.random(10000), dtype=np.float64)
expected = float(np.dot(a_big, b_big))
got = cy_dot(a_big, b_big)
assert abs(got - expected) / expected < 1e-10, f'Ошибка на большом векторе'
print('Все тесты прошли!')

In [ ]:
# БЕНЧМАРК
N = 1_000_000
a = np.ascontiguousarray(np.random.rand(N), dtype=np.float64)
b = np.ascontiguousarray(np.random.rand(N), dtype=np.float64)
a_ls = a.tolist()
b_ls = b.tolist()

t_py = bench(py_dot, a_ls, b_ls)
t_cy = bench(cy_dot, a, b)
t_np = bench(np.dot, a, b)

print(f'dot(N={N:,})')
print(f'  Python : {t_py:.2f} ms  (1.0x)')
print(f'  Cython : {t_cy:.2f} ms  ({t_py/t_cy:.1f}x)')
print(f'  NumPy  : {t_np:.2f} ms  ({t_py/t_np:.1f}x)')
print()
if t_py / t_cy < 5:
    print('Подсказка: добавьте typed memoryview double[::1] к аргументам')
else:
    print('Отлично!')

---
# Задание 4 — 2D Memoryview + `nogil`
**Сложность:** Средний | **Тема:** 2D memoryview, nogil

## Задача
Реализуйте матричное умножение C = A @ B через 2D typed memoryview.

## Что нужно сделать
1. Используйте `double[:, ::1]` для 2D C-contiguous массивов
2. Объявите все переменные цикла через `cdef`
3. Оберните внутренний цикл в `with nogil:`
4. Добавьте директивы компилятора

## Подсказка
```cython
# cython: boundscheck=False
# cython: wraparound=False
import numpy as np

def matmul(double[:, ::1] A, double[:, ::1] B):
    cdef int n = A.shape[0]
    cdef int k = A.shape[1]
    cdef int m = B.shape[1]
    cdef int i, j, p
    cdef double s
    C = np.zeros((n, m), dtype=np.float64)
    cdef double[:, ::1] Cv = C
    with nogil:
        for i in range(n):
            for j in range(m):
                s = 0.0
                for p in range(k):
                    s += A[i, p] * B[p, j]
                Cv[i, j] = s
    return C
```

In [ ]:
import numpy as np

# Эталон на Python
def py_matmul(A, B):
    n, k, m = len(A), len(B), len(B[0])
    C = [[0.0]*m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            s = 0.0
            for p in range(k):
                s += A[i][p] * B[p][j]
            C[i][j] = s
    return C

A = [[1.0, 2.0], [3.0, 4.0]]
B = [[5.0, 6.0], [7.0, 8.0]]
print('Python matmul:')
for row in py_matmul(A, B):
    print(' ', row)  # [19, 22], [43, 50]

In [ ]:
%%cython --quiet
# cython: boundscheck=False
# cython: wraparound=False
# ЗАДАНИЕ 4 — заполните TODO
import numpy as np

def cy_matmul(A, B):  # TODO: добавьте типы double[:, ::1]
    """
    Матричное умножение C = A @ B
    Используйте 2D typed memoryview и nogil
    """
    # TODO: объявите n, k, m, i, j, p, s через cdef
    n = A.shape[0]
    k = A.shape[1]
    m = B.shape[1]
    C = np.zeros((n, m), dtype=np.float64)
    # TODO: объявите Cv как double[:, ::1]
    # TODO: добавьте with nogil:
    for i in range(n):
        for j in range(m):
            s = 0.0
            for p in range(k):
                s += A[i, p] * B[p, j]
            C[i, j] = s
    return C

In [ ]:
# ТЕСТ
import numpy as np
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])
expected = A @ B
got = cy_matmul(A, B)
assert np.allclose(got, expected), f'Ошибка: {got} != {expected}'

# Тест на случайных матрицах
rng = np.random.default_rng(42)
A_big = np.ascontiguousarray(rng.random((50, 60)), dtype=np.float64)
B_big = np.ascontiguousarray(rng.random((60, 40)), dtype=np.float64)
assert np.allclose(cy_matmul(A_big, B_big), A_big @ B_big, rtol=1e-10)
print('Все тесты прошли!')

In [ ]:
# БЕНЧМАРК
rng = np.random.default_rng(42)
N = 128
A_np = np.ascontiguousarray(rng.random((N, N)), dtype=np.float64)
B_np = np.ascontiguousarray(rng.random((N, N)), dtype=np.float64)
A_ls = A_np.tolist()
B_ls = B_np.tolist()

t_py = bench(py_matmul, A_ls, B_ls, repeats=3)
t_cy = bench(cy_matmul, A_np, B_np, repeats=5)
t_np = bench(lambda a, b: a @ b, A_np, B_np, repeats=5)

print(f'matmul({N}x{N})')
print(f'  Python : {t_py:.2f} ms  (1.0x)')
print(f'  Cython : {t_cy:.2f} ms  ({t_py/t_cy:.0f}x)')
print(f'  NumPy  : {t_np:.3f} ms  ({t_py/t_np:.0f}x)')
print()
if t_py / t_cy < 20:
    print('Подсказка: добавьте double[:, ::1] к аргументам и with nogil:')
else:
    print('Отлично! Вы получили C-скорость!')

---
# Задание 5 — Свёртка сигнала (продвинутый)
**Сложность:** Продвинутый | **Тема:** полная оптимизация

## Задача
Реализуйте одномерную свёртку сигнала с ядром (kernel).

```
conv(signal, kernel)[i] = sum(signal[i+j] * kernel[j] for j in range(len(kernel)))
```

Это базовая операция в обработке сигналов, аудио, изображений.

## Что нужно сделать
1. Используйте `double[::1]` для signal и kernel
2. Объявите все переменные через `cdef`
3. Добавьте `with nogil:`
4. Добавьте все директивы: `boundscheck=False`, `wraparound=False`, `cdivision=True`
5. **Бонус:** попробуйте `cdef` вспомогательную функцию для inner loop

## Подсказка
```cython
def convolve(double[::1] signal, double[::1] kernel):
    cdef int n = signal.shape[0]
    cdef int k = kernel.shape[0]
    cdef int out_len = n - k + 1
    cdef int i, j
    cdef double s
    result = np.zeros(out_len, dtype=np.float64)
    cdef double[::1] res = result
    with nogil:
        for i in range(out_len):
            s = 0.0
            for j in range(k):
                s += signal[i + j] * kernel[j]
            res[i] = s
    return result
```

In [ ]:
import numpy as np

# Эталон на Python
def py_convolve(signal, kernel):
    n = len(signal)
    k = len(kernel)
    out_len = n - k + 1
    result = [0.0] * out_len
    for i in range(out_len):
        s = 0.0
        for j in range(k):
            s += signal[i + j] * kernel[j]
        result[i] = s
    return result

signal = [1.0, 2.0, 3.0, 4.0, 5.0]
kernel = [0.5, 0.5]
print('Python convolve:', py_convolve(signal, kernel))
# [1.5, 2.5, 3.5, 4.5]

In [ ]:
%%cython --quiet
# cython: boundscheck=False
# cython: wraparound=False
# cython: cdivision=True
# ЗАДАНИЕ 5 — заполните TODO
import numpy as np

def cy_convolve(signal, kernel):  # TODO: добавьте типы double[::1]
    """
    Одномерная свёртка сигнала с ядром.
    result[i] = sum(signal[i+j] * kernel[j] for j in range(len(kernel)))
    """
    # TODO: объявите n, k, out_len, i, j, s через cdef
    n = signal.shape[0]
    k = kernel.shape[0]
    out_len = n - k + 1
    result = np.zeros(out_len, dtype=np.float64)
    # TODO: объявите res как double[::1]
    # TODO: добавьте with nogil:
    for i in range(out_len):
        s = 0.0
        for j in range(k):
            s += signal[i + j] * kernel[j]
        result[i] = s
    return result

In [ ]:
# ТЕСТ
import numpy as np

# Простой тест
sig = np.ascontiguousarray([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float64)
ker = np.ascontiguousarray([0.5, 0.5], dtype=np.float64)
expected = [1.5, 2.5, 3.5, 4.5]
got = cy_convolve(sig, ker)
assert np.allclose(got, expected), f'Ошибка: {list(got)} != {expected}'

# Сравнение с NumPy
rng = np.random.default_rng(42)
sig_big = np.ascontiguousarray(rng.random(10000), dtype=np.float64)
ker_big = np.ascontiguousarray(rng.random(50), dtype=np.float64)
np_result = np.convolve(sig_big, ker_big, mode='valid')
cy_result = cy_convolve(sig_big, ker_big)
assert np.allclose(cy_result, np_result, rtol=1e-10), 'Ошибка на большом сигнале'
print('Все тесты прошли!')

In [ ]:
# БЕНЧМАРК
rng = np.random.default_rng(42)
N_sig = 100_000
N_ker = 100
sig = np.ascontiguousarray(rng.random(N_sig), dtype=np.float64)
ker = np.ascontiguousarray(rng.random(N_ker), dtype=np.float64)
sig_ls = sig.tolist()
ker_ls = ker.tolist()

t_py = bench(py_convolve, sig_ls, ker_ls, repeats=3)
t_cy = bench(cy_convolve, sig, ker, repeats=5)
t_np = bench(lambda s, k: np.convolve(s, k, mode='valid'), sig, ker, repeats=5)

print(f'convolve(signal={N_sig}, kernel={N_ker})')
print(f'  Python : {t_py:.2f} ms  (1.0x)')
print(f'  Cython : {t_cy:.2f} ms  ({t_py/t_cy:.0f}x)')
print(f'  NumPy  : {t_np:.2f} ms  ({t_py/t_np:.0f}x)')
print()
if t_py / t_cy < 20:
    print('Подсказка: добавьте double[::1], cdef переменные и with nogil:')
else:
    print('Отлично! Вы реализовали свёртку со скоростью C!')

---
# Итоги лабораторной работы

## Что вы освоили

| Задание | Техника | Ожидаемый speedup |
|---------|---------|-------------------|
| 1 — sum_of_squares | `cdef int/long long` | 50-200x |
| 2 — factorial | `cpdef` + `cdef` | 10-50x |
| 3 — dot product | `double[::1]` memoryview | 50-150x |
| 4 — matmul | `double[:, ::1]` + `nogil` | 100-500x |
| 5 — convolve | Полная оптимизация | 100-300x |

## Чеклист оптимизации Cython

```
[ ] 1. Добавить директивы: boundscheck=False, wraparound=False
[ ] 2. Объявить все переменные цикла через cdef
[ ] 3. Использовать typed memoryview вместо list/ndarray
[ ] 4. Добавить with nogil: для чистых C-блоков
[ ] 5. Измерить speedup до и после каждого шага
```

## Связь с проектом

Готовые реализации этих операций находятся в `src/`:
- `src/py_impl/matrix_ops.py` — Pure Python
- `src/cy_typed/matrix_ops_typed.pyx` — Cython typed (эталон)
- `src/c_impl/matrix_ops.c` — Pure C для сравнения

Сгенерированный C-код (после `python setup.py build_ext --inplace`):
- `src/cy_typed/matrix_ops_typed.html` — аннотация (жёлтые строки = Python overhead)